# Harness Engineering Demo: Better Harness Beats Bigger Model

This notebook is built for a management-facing demo.

**Thesis:** a medium model with a good harness can beat a great model with a bad harness, because production quality depends on context, tools, validation, memory, safety gates, observability, and repair loops, not only raw model capability.

We will show the spectrum:

1. **Great model + bad harness**: bare prompt, no shared memory, no runbook enforcement.
2. **Medium model + hand-built harness**: explicit agents, shared memory, sensors, and scorecard.
3. **Medium model + SDK harness**: Strands-style abstraction for tools, hooks, memory, and multi-agent orchestration.
4. **Provider plug-and-play harness**: provider-specific harness lane, represented by DeepSeek adapter boundary.

The reliable part of the demo is deterministic. The live model section calls Ollama Cloud so management can see how this connects to real model backends.

## Demo Architecture

```text
Colab notebook
      ↓
Harness demo repo + Python SDKs
      ↓
Scenario: multi-agent incident response
      ↓
Scorecard: evidence, runbook, safety, memory, completeness
      ↓
Optional live calls to Ollama Cloud
```

Colab is the runtime. Ollama Cloud is the model backend.

## 1. Clone The Repo

Replace `REPO_URL` with your GitHub URL after pushing the project.

If you already uploaded this notebook into the cloned repo, skip this cell and `%cd` into the repo folder.

In [ ]:
REPO_URL = "https://github.com/YOUR_ORG/ollama-harness-engineering-demo.git"

# In Colab, uncomment these two lines after replacing REPO_URL.
# !git clone $REPO_URL
# %cd ollama-harness-engineering-demo

# If you manually uploaded the repo folder or are already in it, this shows the current location.
!pwd
!ls -la

## 2. Install The Demo Dependencies

This happens inside Colab, so it does not depend on your office Mac allowing Python packages.

The important libraries are:

- `strands-agents`: SDK-level harness abstraction.
- `ollama`: direct calls to Ollama Cloud.
- `openai`: useful for OpenAI-compatible endpoints.
- `typer` and `rich`: CLI and readable scorecards.
- `pytest`: quick health check.

In [ ]:
!pip install -r requirements.txt
!pip install -e .

## 3. Verify The Local Demo Works

This section does not call any live model. It proves the harness story reliably before we introduce network/model variability.

In [ ]:
!harness-demo list-lanes

In [ ]:
!harness-demo compare --scenario incident-response

### How To Narrate The Scorecard

- `raw-strong` is the cautionary baseline: a strong model with a bare prompt can sound plausible but fail production controls.
- `hand-built` shows what harness engineering does mechanically: agents, shared memory, runbook checks, safety checks, and scoring.
- `strands-sdk` shows that these controls are moving into reusable SDKs.
- `deepseek-provider` represents the plug-and-play provider-harness direction once provider-specific controls are packaged.

Management takeaway:

> The strategic asset is not one model. The strategic asset is the control layer around models.

## 4. Run Each Lane Individually

This makes the spectrum easier to explain live. Run one cell, pause, explain what changed, then run the next.

In [ ]:
!harness-demo run --scenario incident-response --lane raw-strong

In [ ]:
!harness-demo run --scenario incident-response --lane hand-built

In [ ]:
!harness-demo run --scenario incident-response --lane strands-sdk

In [ ]:
!harness-demo run --scenario incident-response --lane deepseek-provider

## 5. Optional: Run Tests

This is a quick confidence check before the management session.

In [ ]:
!python -m pytest -p no:cacheprovider

# Live Ollama Cloud Section

The next cells call Ollama Cloud directly.

Use this section after the deterministic scorecard. The narrative is:

> The deterministic demo proves the harness architecture. The live model call proves the same idea can route through real cloud-hosted models.

You need an Ollama Cloud subscription/API key.

## 6. Set Ollama Cloud API Key

For a live demo, prefer Colab Secrets if available. The fallback below prompts securely with `getpass`.

Do not hardcode the key in the notebook.

In [ ]:
import os
from getpass import getpass

if not os.environ.get("OLLAMA_API_KEY"):
    os.environ["OLLAMA_API_KEY"] = getpass("Enter OLLAMA_API_KEY: ")

print("OLLAMA_API_KEY configured:", bool(os.environ.get("OLLAMA_API_KEY")))

## 7. Verify Ollama Cloud Connectivity

This uses the official `ollama` Python client against `https://ollama.com`.

In [ ]:
from ollama import Client

ollama_client = Client(
    host="https://ollama.com",
    headers={"Authorization": "Bearer " + os.environ["OLLAMA_API_KEY"]},
)

response = ollama_client.chat(
    model="gpt-oss:20b",
    messages=[{"role": "user", "content": "Reply with exactly: Ollama Cloud connected."}],
    stream=False,
)

print(response["message"]["content"])

## 8. Live Model Comparison: Great Model + Bad Harness vs Medium Model + Good Harness

This is intentionally simple. It shows the core idea without requiring the CLI live lane to be implemented yet.

- Strong model gets a bare incident prompt.
- Medium model gets harnessed context: evidence, runbook, prior memory, required output fields, and safety constraints.

Change the model names if your Ollama account exposes different models.

In [ ]:
STRONG_MODEL = "gpt-oss:120b"
MEDIUM_MODEL = "gpt-oss:20b"

incident_prompt = """
Production checkout latency spiked after a promotion launch.
Payment timeouts increased and customers report failed checkouts.
Investigate and propose a safe next action.
""".strip()

harnessed_prompt = f"""
You are operating inside a production incident-response harness.

Rules:
- Use only supplied evidence, runbook guidance, and shared memory.
- Do not invent facts.
- Avoid destructive actions unless runbook thresholds are met.
- Return these fields: likely_cause, evidence, safe_next_action, rollback_plan, customer_impact, open_questions.

Evidence:
- p95 latency increased from 240ms to 2100ms.
- promotion_price_cache miss events repeated after the promotion launch.
- payment timeouts identify checkout-api as the upstream latency source.

Runbook:
- Enable promotion price cache single-flight lock when miss bursts occur.
- Lower promotion price cache TTL to 60 seconds during rollout.
- Keep payment writes enabled unless timeout rate exceeds 12% for 5 consecutive minutes.
- Prepare rollback to the previous promotion configuration.
- Do not restart all checkout pods unless memory pressure or crash loops are confirmed.

Prior incident memory:
- A similar incident was fixed by enabling single-flight lock and lowering TTL.
- Restarting all checkout pods did not help and worsened cache misses.

Incident:
{incident_prompt}
""".strip()


def ask_ollama(model: str, prompt: str) -> str:
    response = ollama_client.chat(
        model=model,
        messages=[{"role": "user", "content": prompt}],
        stream=False,
    )
    return response["message"]["content"]

print("Configured models:")
print("  Strong/bad harness:", STRONG_MODEL)
print("  Medium/good harness:", MEDIUM_MODEL)

In [ ]:
print("=== Great model + bad harness ===")
raw_answer = ask_ollama(STRONG_MODEL, incident_prompt)
print(raw_answer)

In [ ]:
print("=== Medium model + good harness ===")
harnessed_answer = ask_ollama(MEDIUM_MODEL, harnessed_prompt)
print(harnessed_answer)

## 9. Lightweight Live Answer Review

This is not a perfect evaluator. It is a visible checklist to guide discussion.

For management, the point is not to claim the checklist is magic. The point is that the harness can make quality criteria explicit and repeatable.

In [ ]:
REQUIRED_SIGNALS = {
    "mentions cache issue": ["cache", "promotion_price_cache", "single-flight", "single flight"],
    "uses runbook mitigation": ["single-flight", "single flight", "ttl", "60"],
    "keeps payment writes safe": ["keep payment", "enabled", "12%", "do not disable", "avoid disabling"],
    "has rollback": ["rollback", "previous promotion"],
    "avoids pod restart": ["do not restart", "avoid restart", "not restart"],
}


def checklist(answer: str) -> dict[str, bool]:
    lower = answer.lower()
    return {
        name: any(token.lower() in lower for token in tokens)
        for name, tokens in REQUIRED_SIGNALS.items()
    }


def print_checklist(title: str, answer: str):
    checks = checklist(answer)
    score = sum(checks.values())
    print(title)
    print("score:", score, "/", len(checks))
    for name, passed in checks.items():
        print(f"- {name}: {'yes' if passed else 'no'}")

print_checklist("Great model + bad harness", raw_answer)
print()
print_checklist("Medium model + good harness", harnessed_answer)

## 10. How Strands Fits In The Story

Strands is the middle of the spectrum.

Hand-built harness:

```text
We manually wire agents, tools, memory, hooks, and sensors.
```

Strands SDK harness:

```text
The framework gives us agent runtime concepts: tools, hooks, context managers, session memory, observability, and multi-agent patterns.
```

Provider harness:

```text
Some provider-specific behavior becomes plug-and-play behind an adapter.
```

Management message:

> The industry is moving from handcrafted harnesses toward reusable agent infrastructure.

## 11. Suggested Closing Slide

**Title:** Harness Engineering Is The Control Plane For AI Applications

```text
Model capability helps.
Harness quality decides whether the result is usable in production.
```

What the harness gives us:

- controlled context
- shared memory
- tool boundaries
- safety gates
- validation and repair
- observability
- replaceable models
- lower-cost model options

Final message:

> We should evaluate AI applications by the strength of their harness, not just the size of their model.

# Notes For Iteration

After you run this in Colab, capture what management reacts to:

- Do they understand the four lanes?
- Is `raw-strong` too harsh at `0/100`?
- Should the demo include costs/tokens in the scorecard?
- Should we make the live model outputs feed into the same deterministic scorer?
- Should we add a real Strands implementation cell instead of the current repo lane abstraction?

Those are the next high-value improvements.